## Unsupervised clustering on Emotify (MP3)

- Load the first **N** tracks per genre folder under `emotifymusic/` (four folders: classical, electronic, pop, rock).
- Extract the same tabular features as in `jaschacomments_5s.ipynb`: MFCC, chroma, spectral contrast, and ZCR summaries (`extract_features`).
- **MP3 vs WAV:** `librosa.load` decodes via `audioread` (often backed by ffmpeg). If loading fails, install ffmpeg and retry.
- **K-Means (k = 5):** For *k* unsupervised groups we use **K-Means** with `n_clusters=5`, matching the five mood classes in the labelled Kaggle set.
- Fit a `StandardScaler` + `KMeans` on the emotify feature matrix and save a small artifact bundle for `compare.ipynb`.

In [1]:
import os
from pathlib import Path

import joblib
import librosa
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path(".").resolve()
EMOTIFY_ROOT = BASE_DIR / "/content/drive/MyDrive/thesis/emotifymusic"
ARTIFACTS_DIR = BASE_DIR / "/content/drive/MyDrive/thesis/artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
CLUSTERING_ARTIFACT_PATH = ARTIFACTS_DIR / "emotify_kmeans_tabular.joblib"

# How many MP3 files to take from each genre folder (in numeric filename order).
SONGS_PER_EMOTIFY_GENRE = 100

N_MOOD_CLUSTERS = 5  # same as aggressive / dramatic / happy / romantic / sad

KMEANS_RANDOM_STATE = 42

Mounted at /content/drive


In [2]:
def extract_features(file_path):
    """MFCC, chroma, spectral contrast, and ZCR summaries (same as jaschacomments_5s)."""
    try:
        y, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_mean = np.mean(mfcc.T, axis=0)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        chroma_mean = np.mean(chroma.T, axis=0)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        contrast_mean = np.mean(contrast.T, axis=0)
        zcr = librosa.feature.zero_crossing_rate(y)
        zcr_mean = np.mean(zcr)
        return np.hstack((mfcc_mean, chroma_mean, contrast_mean, zcr_mean))
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


def natural_mp3_sort_key(filename: str):
    """Sort 1.mp3, 2.mp3, …, 10.mp3 numerically when the stem is digits."""
    stem = Path(filename).stem
    return (0, int(stem)) if stem.isdigit() else (1, stem)


def list_emotify_genre_folders(root: Path):
    subs = [p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")]
    return sorted(subs, key=lambda p: p.name.lower())


def collect_emotify_paths(root: Path, n_per_folder: int):
    paths = []
    genre_labels = []
    for folder in list_emotify_genre_folders(root):
        mp3s = sorted(
            [f for f in os.listdir(folder) if f.lower().endswith(".mp3")],
            key=natural_mp3_sort_key,
        )[:n_per_folder]
        for fn in mp3s:
            paths.append(str((folder / fn).resolve()))
            genre_labels.append(folder.name)
    return paths, genre_labels

In [3]:
paths, genre_labels = collect_emotify_paths(EMOTIFY_ROOT, SONGS_PER_EMOTIFY_GENRE)
print(f"Collected {len(paths)} files from {EMOTIFY_ROOT}")
if len(paths) == 0:
    raise FileNotFoundError(f"No MP3s found under {EMOTIFY_ROOT}")

features = []
ok_paths = []
ok_genres = []
for fp, g in zip(paths, genre_labels):
    vec = extract_features(fp)
    if vec is not None:
        features.append(vec)
        ok_paths.append(fp)
        ok_genres.append(g)

X_emotify = np.asarray(features, dtype=np.float32)
print("Feature matrix:", X_emotify.shape, "(expect 33-dim vectors)")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_emotify)

kmeans = KMeans(
    n_clusters=N_MOOD_CLUSTERS,
    random_state=KMEANS_RANDOM_STATE,
    n_init="auto",
)
cluster_labels = kmeans.fit_predict(X_scaled)

print("Cluster counts:", np.bincount(cluster_labels, minlength=N_MOOD_CLUSTERS))

Collected 400 files from /content/drive/MyDrive/thesis/emotifymusic
Feature matrix: (400, 33) (expect 33-dim vectors)
Cluster counts: [ 65  89  92  48 106]


In [4]:
bundle = {
    "kmeans": kmeans,
    "scaler": scaler,
    "X_emotify": X_emotify,
    "paths": ok_paths,
    "genre_labels": ok_genres,
    "cluster_labels": cluster_labels,
    "meta": {
        "SONGS_PER_EMOTIFY_GENRE": SONGS_PER_EMOTIFY_GENRE,
        "N_MOOD_CLUSTERS": N_MOOD_CLUSTERS,
        "EMOTIFY_ROOT": str(EMOTIFY_ROOT),
        "KMEANS_RANDOM_STATE": KMEANS_RANDOM_STATE,
    },
}
joblib.dump(bundle, CLUSTERING_ARTIFACT_PATH)
print("Saved:", CLUSTERING_ARTIFACT_PATH)

Saved: /content/drive/MyDrive/thesis/artifacts/emotify_kmeans_tabular.joblib
